# Control

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alx87grd/minilink/blob/main/examples/tutorial/03_control.ipynb)

Official API intro to controllers and design helpers: impedance / proportional /
filtered loops, LQR → `StateFeedback`, an observer and LQG for a
measured output, and a peek at model-based methods.

**Next:** [`04_analysis`](04_analysis.ipynb)


In [ ]:
# Local conda: minilink already installed. Colab: clone + path + meshcat.
import sys

if "google.colab" in sys.modules:
    get_ipython().run_line_magic("matplotlib", "inline")
    get_ipython().system("git clone https://github.com/alx87grd/minilink")
    sys.path.insert(0, "/content/minilink")
    get_ipython().system("pip install -q meshcat")

## LQR at an operating point

`minilink.dynamics.catalog` ships ready-to-use plants; `minilink.control` ships
ready-to-wire controllers and design factories. Below: catalog sweep, forced
cart-pole response, then LQR stabilization.


In [ ]:
import numpy as np
from minilink import CartPole, Drone2D, KinematicCar

for plant in (CartPole(), KinematicCar(), Drone2D()):
    print(f"{plant.name:<22} n={plant.n}  m={plant.m}  states={plant.state.labels}")

cartpole = CartPole()
cartpole.compute_forced(lambda t: np.array([3.0 * np.sin(2.0 * t)]))
cartpole.plot_trajectory()

## Control catalog

Controllers are `System` blocks with standard ports (`r`, `y`, `u`).
The quick start used `Impedance`; the catalog also includes
`IntegralImpedance`, `PID`, and the design verbs such as
`lqr(plant, Q, R, x_bar=x_bar)` — linearize at an equilibrium and return a
`StateFeedback` ready to wire with `@`.


In [ ]:
import numpy as np
from minilink import CartPole, lqr, lqr_gain

plant = CartPole()
x_bar = np.array([0.0, np.pi, 0.0, 0.0])
Q = np.diag([1.0, 1.0, 1.0, 1.0])
R = np.array([[1.0]])

# The design chain in the open: A = df/dx, B = df/du, then the Riccati gain.
A, B = plant.jacobian("f", "x", x_bar), plant.jacobian("f", "u", x_bar)
print("A =\n", np.round(A, 3))
K = lqr_gain(A, B, Q, R)
print("K =", np.round(K, 3))

# One line does the same and returns the controller trimmed about (x_bar, u_bar).
lqr_ctl = lqr(plant, Q, R, x_bar=x_bar)
lqr_loop = lqr_ctl @ plant

plant.x0 = np.array([-3.0, np.pi - 0.3, 0.0, 0.0])
lqr_loop.compute_trajectory(tf=8.0)
lqr_loop.plot_trajectory()


## Observer and LQG

The LQR above reads the full state `x`; a sensor gives `y`, here the cart and pole
positions only. `kalman` designs the observer that rebuilds `x` from `u` and `y`, and
`lqg` puts the LQR gain on its estimate: one controller box, `y` in and `u` out, closed
on the plant with `@`.

In [ ]:
from minilink import Gain, kalman, lqg

# A sensor measures the two positions only: it is part of the plant box
cartpole = CartPole()
cartpole.x0 = np.array([-1.0, np.pi - 0.2, 0.0, 0.0])
sensed = cartpole >> Gain(np.eye(4)[:2])
Q_n = 1e-2 * np.eye(4)  # process-noise intensity on the state
R_n = 1e-4 * np.eye(2)  # measurement-noise intensity on the positions

# The observer rebuilds x from u and y; print shows its gain L and the poles of A - L C
observer = kalman(sensed, Q_n, R_n, x_bar=x_bar)
print(observer)

# The LQR gain on the Kalman estimate, in one controller box
lqg_loop = lqg(sensed, Q, R, Q_n, R_n, x_bar=x_bar) @ sensed
trajectory = lqg_loop.compute_trajectory(tf=8.0)

In [ ]:
lqg_loop.plot_trajectory()

## Model-based peek


In [ ]:
from minilink import ComputedTorque, SlidingMode

print(ComputedTorque.__name__, SlidingMode.__name__)